<img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/9feba6889d498d8c754a6783e5460f46e054d60f/assets/banner-level2.png" width="100%" alt="AAAR 2026 photochemical box modeling tutorial. Level 2 of 3: Limonene SOA. How does limonene chemistry make particle mass? Supported by Reflective.">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/notebooks/02_limonene_soa.ipynb)

In this notebook 10 ppb of limonene is oxidized by O₃, OH and NO₃ in a box under constant,
chamber-like light. The gas-phase chemistry is explicit: the Master Chemical Mechanism (MCM v3.3.1)
limonene subset, with about 725 species and 2135 reactions, solved as a stiff system of ODEs. Each
closed-shell organic product gets a saturation concentration c\*. The products are lumped into
half-decade volatility classes, and the classes partition between the gas and a seed aerosol. That
is the Level 1 physics, now fed by real chemistry. The Limonene SOA tab of the web app runs the same
model and preset with sliders, on the same 40 size bins. Here you can read the mechanism, change any input and write your own
analysis.

> <font color="#8B8D8E"><b>CAVEAT</b></font> &nbsp; The saturation concentration c\* of each product comes from a per-species table supplied by Shantanu Jathar (Colorado State University). He computed the vapour pressure of every product at 298.15 K with the group-contribution method of [Nannoolal et al. (2008)](https://doi.org/10.1016/j.fluid.2008.04.020), using the box model PyCHAM 5.8.0 ([O'Meara et al., 2021](https://doi.org/10.5194/gmd-14-675-2021)). An estimate like this can be off by about a decade for a single molecule, and other methods can differ by more for molecules with several functional groups. <mark>c\* is still the most uncertain input here: shifting every c\* up by one decade removes about three quarters of the SOA (Section 4).</mark> The full list of model assumptions is in [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/0e23da0da79a5a080baa02447e47951b75f796fe/ASSUMPTIONS.md); A8 (c\*) and A10 (constant light) matter most here.

> <font color="#8B8D8E"><b>HOW TO USE IT</b></font> &nbsp; Run the cells from top to bottom (Shift+Enter). Boxes labelled <font color="#A47F52"><b>TRY THIS</b></font> name a variable to change: edit it, run that cell again, then the plot cells below it. On Colab most of those cells also show a title and an input box; editing the box changes the code. Comments that start with `# Try this` suggest changes in other cells. All plots are interactive: hover for values, drag to zoom, double-click to reset, and click a legend entry to hide or show a line; where a plot has Linear/Log buttons (top right), they switch the scale. The *Size resolution* cell below sets how many size bins every run uses.

<details>
<summary><b>Run times</b></summary>

The first model run in a session also compiles the model (JAX) and builds the photolysis table. That adds about 5 s on a laptop, more on Colab. After that a 6-hour Level 2 run takes about 6–8 s on a laptop, and maybe twice that on Colab. The whole notebook takes about 1.5 minutes on a laptop and about 3 minutes on Colab.

</details>

## Setup

> <font color="#8B8D8E"><b>SETUP</b></font> &nbsp; Run the next two cells once per session.
>
> - **Colab:** the first cell installs the box model and the tutorial helpers (1–2 minutes). Red pip
>   "dependency conflicts" lines are expected. If a cell asks you to restart, click
>   *Runtime › Restart session*, then *Runtime › Run all*.
> - **Your own computer:** install the repository first ([README](https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/README.md#run-on-your-own-computer));
>   the first cell then does nothing.
> - `at.setup()` downloads the TUV-x photolysis data once (about 5 MB) and prints the versions.

In [ ]:
# Colab: installs the tutorial helpers and the box model (first time: 1-2 minutes).
# Red pip "dependency conflicts" lines on Colab are expected and do not matter here.
import importlib.metadata
import sys

HELPERS_VERSION = "0.4.7"   # the aaar_tutorial version these notebooks need


def _older(version):
    return tuple(map(int, version.split("."))) < tuple(map(int, HELPERS_VERSION.split(".")))


try:
    installed = importlib.metadata.version("aaar-tutorial")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or _older(installed):
    if "google.colab" not in sys.modules:
        raise ModuleNotFoundError(
            f"This notebook needs the tutorial helpers {HELPERS_VERSION}, found {installed or 'none'}. Pick the "
            "kernel of the repository's .venv, update it (git pull, then uv sync or pip install -e .), then "
            "restart the kernel.")
    %pip install -q "aaar-tutorial @ git+https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial@main"
    importlib.invalidate_caches()           # so the next cell finds the new packages
loaded = getattr(sys.modules.get("aaar_tutorial"), "__version__", None)
if loaded is not None and _older(loaded):
    raise RuntimeError("This session still has older tutorial helpers loaded. Colab: Runtime > Restart "
                       "session, then Runtime > Run all. Locally: Kernel > Restart Kernel.")

In [ ]:
import numpy as np
import plotly.graph_objects as go

import aaar_tutorial as at
from aaar_tutorial import config, plots

at.setup()
PRESET = "level2_limonene_soa"

In [ ]:
#@title Size resolution
# Size bins for every run in this notebook. 40 (the web app's grid) is the fastest; 80, 160 and 320
# draw the size distribution more finely (results change by under 1 %) and take longer: a Level 1
# run takes about 1, 4, 14 and 60 s on a laptop, Level 3 about 12, 19, 39 and 140 s; Colab is
# about twice as slow. Rerun the cells below after changing it.
N_BINS = 40  #@param [40, 80, 160, 320] {type:"raw"}
at.use_bins(N_BINS)

## Learning goals

> <font color="#8B8D8E"><b>LEARNING GOALS</b></font> &nbsp; After this notebook you can:
>
> 1. Say what an explicit mechanism contains (species, reactions, rate constants) and look up a reaction in it.
> 2. Work out which oxidant (O₃, OH or NO₃) removes limonene in this box, and why.
> 3. Explain how a product gets a c\* and is lumped into a half-decade volatility class.
> 4. Use absorptive partitioning, $F = 1/(1 + c^*/C_{OA})$, to read the volatility plot and check the model against it.
> 5. Read the top SOA contributors and explain why most of them are hydroperoxides.
> 6. Change NOx, O₃, limonene and the seed, compare the runs, and say which conclusions depend on c\*.

## 1. The scenario

The preset is a 6-hour run at 298 K and 50% RH with:

- 10 ppb limonene, 40 ppb O₃, 0.5 ppb NO, 2 ppb NO₂ and 1 ppb SO₂, plus background CO, CH₄ and H₂;
- a seed of 2000 particles cm⁻³ at 80 nm, 30% non-volatile organic by mass (the rest sulfate);
- **constant light** (chamber-like): the TUV-x spectrum at a solar zenith angle of 30°, scaled so
  that J(NO₂) = 5×10⁻³ s⁻¹ (ASSUMPTIONS A10).

> <font color="#8B8D8E"><b>THE BOX</b></font> &nbsp; There is no dilution, deposition or wall loss.

`at.describe` prints every input you can change, grouped by section:

In [ ]:
at.describe(PRESET)

### Run the preset

`at.run` runs the model and shows a progress bar. The first run in a session also compiles the
model. If you are short on time, set `USE_PRECOMPUTED = True` to load a saved result of the
unchanged preset instead (instant). The experiments further down always run the model.

In [ ]:
#@title Run the preset
# True: load the saved result of the unchanged preset (no model run)
USE_PRECOMPUTED = False  #@param {type:"boolean"}

if USE_PRECOMPUTED:
    base = at.load_example(PRESET)
else:
    base = at.run(PRESET, name="preset")

for key, value in at.summary(base).items():
    print(f"{key:>15}  {value:.4g}")

## 2. Explicit gas-phase chemistry (MCM)

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; An explicit mechanism lists every reaction of every species. A reaction A + B → products runs at $r = k\,[A][B]$, with $k$ in cm³ molec⁻¹ s⁻¹ (or $r = J[A]$ for photolysis, with $J$ in s⁻¹). Each species then gets one ODE, production minus loss:
>
> $$\frac{d[X_i]}{dt} = \sum_j \nu^{+}_{ij}\, r_j \;-\; \sum_j \nu^{-}_{ij}\, r_j , \qquad r_j = k_j\,[A_j]\,[B_j]$$
>
> where $\nu^{+}_{ij}$ and $\nu^{-}_{ij}$ are how many molecules of species $i$ reaction $j$ makes and uses.

The lifetimes in this system range from fractions of a second (O¹D, OH) to months (CO), so
the equations are *stiff*. The model integrates them with an implicit, adaptive solver (diffrax
Kvaerno3) over 300-s coupling intervals (60 s if you switch nucleation on, which also shortens the
aerosol steps from 60 s to 5 s to follow a nucleation burst and makes a run take about 25 s instead
of 6–8 s). As in KPP (the Kinetic PreProcessor, a standard tool and
file format for chemical mechanisms), it holds every $k$ fixed within an interval (ASSUMPTIONS A12).

The mechanism file is the MCM v3.3.1 limonene subset in KPP format, and the model parses it
directly. `at.mechanism` parses it and evaluates every rate constant at the preset's temperature,
pressure and RH ([Notebook 4](04_inside_the_model.ipynb) shows how). Let's open it:

In [ ]:
chem = at.mechanism(PRESET)   # the mechanism, with rate constants k at the preset's T, p and RH
mech = chem.mechanism         # the .eqn file (725 species) plus the N2O5-uptake counter NA_HET (A1)

print(f"species:              {len(mech.species)}")
print(f"reactions:            {len(mech.reactions)}")
print(f"photolysis reactions: {sum(r.photolysis for r in mech.reactions)}")
print(f"peroxy radicals RO2:  {len(mech.ro2)}")

In [ ]:
chem.show_reactions("LIMONENE")   # Try this: "O3", "NO2", or any name from mech.species

Limonene has three sets of first steps: OH addition (three branches, scaled to the chamber-measured
k_OH = 1.47×10⁻¹⁰ cm³ molec⁻¹ s⁻¹; A4), ozonolysis (two Criegee intermediates) and NO₃ addition.
Each gives a peroxy radical (RO₂) or a Criegee intermediate. Hundreds of later reactions turn these
into closed-shell products: carbonyls, acids, hydroperoxides, nitrates and PANs.

**Follow one product.** The hydroperoxide LIMALOOH is the largest SOA contributor in the preset
(Section 5). Where does it come from, and where does it go?

In [ ]:
print("Made by:")
chem.show_reactions("LIMALOOH", role="product")
print("\nLost by:")
chem.show_reactions("LIMALOOH")   # Try this: "LIMALO2" (its peroxy radical), "C822PAN" (a PAN) or "LIMANO3" (a hydroxy nitrate)

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; A hydroperoxide (–OOH) forms when a peroxy radical meets HO₂. Here the radical is LIMALO₂, which OH makes from limonaldehyde (LIMAL). The same radical can react with NO instead, so NO competes with HO₂ for it; Section 6a shows what that does to the SOA. In the gas phase, OH turns LIMALOOH back into LIMALO₂, which either forms LIMALOOH again with HO₂ or reacts with NO and is lost, and light splits it. Against OH alone its lifetime is $\tau = 1/(k_{OH}[\mathrm{OH}])$, where $k_{OH}$ = 4.65×10⁻¹¹ cm³ molec⁻¹ s⁻¹ is the rate constant printed above and [OH] the OH concentration. At this run's mean OH of 7.8×10⁵ cm⁻³ that is about 8 hours. In full sunlight, with OH near 1.2×10⁷ cm⁻³ (the first afternoon of Level 3), it is about half an hour.
>
> PANs behave differently. A PAN forms when an acyl peroxy radical meets NO₂ and falls apart again by heat: its decomposition rate of about 4×10⁻⁴ s⁻¹ at 298 K means a gas-phase lifetime of about 40 minutes (try C822PAN in the cell above).

### Oxidants and gases over time

> <font color="#A47F52"><b>EXPLORE</b></font> &nbsp; Click legend entries to show or hide species. Look at how fast limonene is used up, what happens to NO and NO₂, and the levels of OH, HO₂ and NO₃. Light is constant, so there is no day–night cycle.

In [ ]:
plots.gases(base)

In [ ]:
plots.radicals(base)

### Who oxidizes limonene?

The rate at which oxidant X removes limonene is $L_X = k_X\,[X]\,[\mathrm{LIM}]$. Integrating it
over the run gives each oxidant's share of the limonene removed:

$$\text{share}_X = \frac{\int k_X [X][\mathrm{LIM}]\,dt}{\sum_Y \int k_Y [Y][\mathrm{LIM}]\,dt}$$

The cell below takes $k_X$ from the mechanism and $[X]$ and $[\mathrm{LIM}]$ from the model
output, which is saved every 10 minutes. The integral is therefore approximate.

In [ ]:
result = base   # Try this: a run from Section 6, e.g. nox_runs[-1] or o3_runs[-1]

k_limonene = {}   # [cm3 molec-1 s-1], summed over the branches of each oxidant
for r in mech.reactions:
    if "LIMONENE" in r.reactants and len(r.reactants) == 2:
        oxidant = r.reactants[1 - r.reactants.index("LIMONENE")]
        k_limonene[oxidant] = k_limonene.get(oxidant, 0.0) + chem.k[r.label]

t_h = np.asarray(result.time_h)
ppb_to_cm3 = chem.air["M"] * 1e-9
oxidant_cm3 = {"O3": at.series(result, "O3") * ppb_to_cm3,
               "OH": at.series(result, "OH"),
               "NO3": at.series(result, "NO3")}
limonene_ppb = at.series(result, "LIMONENE")

fig = go.Figure()
removed_ppb = {}
for x, conc in oxidant_cm3.items():
    loss_ppb_per_h = k_limonene[x] * conc * limonene_ppb * 3600.0
    removed_ppb[x] = np.trapezoid(loss_ppb_per_h, t_h)
    fig.add_trace(go.Scatter(x=t_h, y=loss_ppb_per_h, name=f"{x}  (k = {k_limonene[x]:.2g})"))
fig.update_xaxes(title=config.TIME_AXIS_TITLE)
fig.update_yaxes(title="Loss rate (ppb h⁻¹)")

total = sum(removed_ppb.values())
print("Share of limonene removed:", ", ".join(f"{x} {100 * v / total:.0f}%" for x, v in removed_ppb.items()))
print(f"Sum of the integrals: {total:.2f} ppb; limonene used in the model: "
      f"{limonene_ppb[0] - limonene_ppb[-1]:.2f} ppb")
plots.style(fig, "Limonene loss rate by oxidant")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; Per molecule, OH reacts with limonene about 700 000 times faster than O₃. But O₃ (40 ppb, or 10¹² molec cm⁻³) is more than a million times more abundant than OH (a few 10⁵ cm⁻³), so here O₃ removes about 60% of the limonene and OH most of the rest. NO₃ matters little under constant light because it is photolysed within seconds. At night it would matter much more (Level 3). The two totals printed above differ by about 10% because limonene falls fast early on, and 10-minute outputs are too coarse to integrate that exactly.

## 3. From products to volatility classes

Only **closed-shell** organic products partition to the particles. Radicals (RO₂, RO, acyl peroxy)
react too fast to matter. The model finds the closed-shell species by valence parity: the number of
H + N atoms is even. Each one gets a c\* from Shantanu Jathar's table. For every product, PyCHAM
estimated the pure-component vapour pressure $p^\circ$ at 298.15 K with the method of
[Nannoolal et al. (2008)](https://doi.org/10.1016/j.fluid.2008.04.020), and the table converts it to a saturation concentration:

$$c^* = \frac{10^6\,M\,p^\circ}{R\,T}$$

Here $c^*$ is in µg m⁻³, $M$ is the molar mass (g mol⁻¹), $p^\circ$ the vapour pressure (Pa),
$R$ = 8.314 J mol⁻¹ K⁻¹ the gas constant, $T$ = 298.15 K, and 10⁶ converts grams to micrograms.
Nannoolal et al. build the vapour pressure from the molecule's structure: each structural group
adds its own fitted contribution, and some pairs of groups add an interaction term. The method
also needs the normal boiling point, itself estimated from the structure; which boiling-point
estimate PyCHAM used for this table is not recorded (ASSUMPTIONS A8). The model uses the table's values as its c\* at 298 K (ASSUMPTIONS A8).

The products are lumped into 15 half-decade classes centred on log₁₀ c\* = −3, −2.5, …, 4, each
±0.25 decade wide. Inside the model each product condenses with the c\* of its class centre, so
it can be treated as up to a quarter decade (a factor of about 1.8) more or less volatile than
the table says; LIMALOOH (10^0.14), for example, sits in the class at 10^0. Compared with PyCHAM
run with the same vapour pressures, this lumping is the largest reason the model's SOA is about
9 % higher after 6 hours. Products more volatile than 10^4.25 µg m⁻³ stay in the gas phase, and anything
below 10^−3.25 would join the lowest class (the table has nothing that low). For the temperature
dependence each class gets ΔH_vap = 131 − 11 log₁₀ c\* kJ mol⁻¹
([Epstein et al., 2010](https://doi.org/10.1021/es902497z)).

Four closed-shell carbon species are not in the table: limonene, CO, CH₄ and the OH tracer D9. They
get the estimate of [Li et al. (2016)](https://doi.org/10.5194/acp-16-3327-2016) described below, which puts all four far
above the top class, so none of them condenses.

<details>
<summary><b>A comparison: the composition-only estimate of Li et al. (2016)</b></summary>

Before the table arrived, the model estimated c\* at 298 K from the elemental composition alone:

$$\log_{10} c^* = (n_C^0 - n_C)\,b_C \;-\; n_O\,b_O \;-\; 2\,\frac{n_C\,n_O}{n_C + n_O}\,b_{CO}
\;-\; n_N\,b_N \;-\; n_S\,b_S$$

$n_C$, $n_O$, $n_N$ and $n_S$ are atom counts, and the coefficients are fitted separately for each
composition class (CH, CHO, CHON, …; [Li et al., 2016](https://doi.org/10.5194/acp-16-3327-2016), Table 1). The formula only
counts atoms. It cannot tell a thermally labile PAN (–C(O)OONO₂) from a stable nitrate (–ONO₂), and
it does not see functional groups at all. Over the 458 closed-shell organics in the table, the table
is at the median 2.7 decades more volatile than this formula for the nitrogen-containing species,
and 0.4 decades less volatile for the others. With the formula, almost all of the Level 2 SOA was
PANs and organic nitrates. You can still run it (Section 5) and compare.

</details>

In [ ]:
from tomas_jax.gaschem.config import CSTAR_TABLE_SOURCES, MECHANISM_DIR
from tomas_jax.gaschem.volatility import assign_classes, is_closed_shell, load_cstar_table, log10_cstar_li2016

table_name = at.scenario_from(PRESET).cstar_table        # the preset's c* table file
table = load_cstar_table(MECHANISM_DIR / table_name)      # {species: log10 c* at 298 K}

examples = {
    "LIMONENE": "the precursor",
    "LIMAL": "limonaldehyde, a first-generation product",
    "LIMONONIC": "limononic acid",
    "LIMALOOH": "a hydroperoxide (LIMALO2 + HO2), the top SOA contributor in the preset",
    "LIMANO3": "a hydroxy nitrate (LIMAO2 + NO)",
    "C822PAN": "a PAN",
}   # Try this: add any species from mech.species
print(f"{'':22} log10 c* (µg/m³)\n{'species':10} {'formula':11} {'table':>5}   {'Li 2016':>7}")
for name, what in examples.items():
    in_table = f"{table[name]:5.2f}" if name in table else "    –"
    print(f"{name:10} {chem.formula(name):11} {in_table}   {log10_cstar_li2016(mech.atoms[name]):7.2f}   {what}")

classes = assign_classes(mech, table, CSTAR_TABLE_SOURCES.get(table_name, table_name))   # the model's lumping
print(f"\n{len(classes.species)} condensable species in {classes.class_cstar.size} classes; "
      f"c* source: {classes.source}")
print("Not in the table (Li et al. 2016 used):", ", ".join(classes.fallback))

The histogram shows the c\* of every closed-shell organic in the mechanism, as the model uses it
(the table, plus Li et al. 2016 for the four species it lacks). Click the two *Li et al. 2016*
entries in the legend to compare with the composition-only estimate. The grey band is the range
the partitioning classes cover, and the dashed line is the absorbing organic mass C_OA at the end
of the preset run. Species to the left of that line end up mostly in the particles.

In [ ]:
closed_shell = [s for s in mech.species if mech.atoms[s].get("C") and is_closed_shell(mech.atoms[s])]
log_cstar_li = {s: log10_cstar_li2016(mech.atoms[s]) for s in closed_shell}
log_cstar = {s: table.get(s, log_cstar_li[s]) for s in closed_shell}   # what the model uses
has_n = {s: bool(mech.atoms[s].get("N")) for s in closed_shell}

fig = go.Figure()
for values, source, visible, pattern in ((log_cstar, "table", True, ""),
                                         (log_cstar_li, "Li et al. 2016", "legendonly", "/")):
    for flag, label, color in ((False, "no N", config.PARTICLE_COLOR),
                               (True, "contains N (nitrates, PANs)", config.SERIES_COLORS[1])):
        fig.add_trace(go.Histogram(x=[values[s] for s in closed_shell if has_n[s] == flag],
                                   name=f"{source}: {label}", visible=visible,
                                   xbins=dict(start=-6.25, end=12.25, size=0.5), marker_color=color,
                                   marker_pattern_shape=pattern, opacity=0.7))
edge = 0.25   # half-width of a class [decades]
fig.add_vrect(x0=np.log10(classes.class_cstar.min()) - edge,
              x1=np.log10(classes.class_cstar.max()) + edge, fillcolor="rgba(0,0,0,0.07)",
              line_width=0, layer="below", annotation_text="partitioning classes",
              annotation_position="top left")
c_oa_end = at.series(base, "SOA_ugm3")[-1] + at.series(base, "POA_ugm3")[-1]
fig.add_vline(x=np.log10(c_oa_end), line=dict(color="#444444", dash="dash"),
              annotation_text=f"C_OA at 6 h ≈ {c_oa_end:.0f} µg m⁻³", annotation_position="top right")
fig.update_layout(barmode="overlay")
fig.update_xaxes(title="log₁₀ c* at 298 K (µg m⁻³)")
fig.update_yaxes(title="Number of species")
plots.style(fig, "c* of the closed-shell organics in the mechanism")

## 4. Absorptive partitioning

> <font color="#8B8D8E"><b>KEY IDEA</b></font> &nbsp; The rule is the same as in Level 1, now applied to each class $j$. At equilibrium the particle fraction is
>
> $$F_j = \frac{C_{p,j}}{C_{p,j} + C_{g,j}} = \frac{1}{1 + c^*_j / C_{OA}}, \qquad C_{OA} = C_{nv} + \sum_j F_j\,\bigl(C_{p,j} + C_{g,j}\bigr)$$
>
> $C_{OA}$ is the absorbing organic mass: the SOA plus the seed's non-volatile organic ($C_{nv}$, 0.77 µg m⁻³). Sulfate, nitrate and water do not absorb organics in this model. A class with $c^* = C_{OA}$ is half in the particles.

The model does not impose this equilibrium. It computes condensation and evaporation kinetically for every size bin, with Raoult's law (by organic mass fraction, the same assumption as $C_{OA}$ in the equation above; by mole fraction the SOA would be 1–2 % lower) and the Kelvin effect, and so relaxes towards it.

> <font color="#A47F52"><b>EXPLORE</b></font> &nbsp; Move the time slider in the volatility plot. As C_OA grows (the dashed line moves right), more of each class moves into the particles.

In [ ]:
plots.volatility(base)

**Check the model against the equation.** For every class that holds mass, the cell compares the
model's particle fraction with $1/(1 + c^*/C_{OA})$, using the model's own C_OA. The table's
values are for 298.15 K, but the model treats them as its 298 K reference and corrects them to the
run temperature of 298.15 K, so the c\* it applies is 2–3 % above the table (ASSUMPTIONS A8). The
cell draws the curve with the 298 K values, which accounts for part of the small gap below the
curve.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `time_index` to an earlier output, e.g. 1 (10 min), 6 (1 h) or 18 (3 h); outputs are every 10 min.

In [ ]:
#@title Choose the output time
# index of the 10-minute outputs; -1 is the end of the run
time_index = -1  #@param {type:"raw"}

vol = base.volatility
log10_cstar = np.asarray(vol["log10_cstar"])
particle = np.asarray(vol["particle_ugm3"])[time_index]
gas = np.asarray(vol["gas_ugm3"])[time_index]
c_nv = at.series(base, "POA_ugm3")[time_index]
c_oa = at.series(base, "SOA_ugm3")[time_index] + c_nv

has_mass = particle + gas > 1e-3                 # skip empty classes
fraction = particle[has_mass] / (particle + gas)[has_mass]
x_line = np.linspace(-3.5, 4.5, 200)
fig = go.Figure([
    go.Scatter(x=x_line, y=1 / (1 + 10**x_line / c_oa), name=f"equilibrium, C_OA = {c_oa:.2f} µg m⁻³",
               line=dict(color="#444444", dash="dash")),
    go.Scatter(x=log10_cstar[has_mass], y=fraction, mode="markers",
               name="model", marker=dict(size=10, color=config.PARTICLE_COLOR)),
])
fig.update_xaxes(title="log₁₀ c* at 298 K (µg m⁻³)")
fig.update_yaxes(title="Particle fraction F")
plots.style(fig, f"Particle fraction per c* class at t = {base.time_h[time_index]:g} h")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; The model points sit on or just below the equilibrium curve. Vapours are still being made, so the particles lag a little behind; the 298 K c\* values used here add a little to the gap.

**How sensitive is the SOA to c\*?** If we keep the total (gas + particle) mass of each class from
the run and shift every c\* by the same number of decades, the equilibrium equation gives a new
SOA. The cell solves $C_{OA} = C_{nv} + \sum_j C_{tot,j} / (1 + c^*_j/C_{OA})$ by fixed-point
iteration. It needs no model run, but it ignores how the chemistry would respond.

In [ ]:
def equilibrium_soa(total_ugm3, log10_cstar, c_nv, shift_decades=0.0):
    """SOA [ug/m3] at absorptive equilibrium for class totals `total_ugm3` and c* shifted by `shift_decades`."""
    cstar = 10.0 ** (log10_cstar + shift_decades)
    c_oa = c_nv + total_ugm3.sum()                 # start from "everything condenses"
    for _ in range(1000):
        new = c_nv + np.sum(total_ugm3 / (1 + cstar / c_oa))
        if abs(new - c_oa) < 1e-9 * c_oa:
            break
        c_oa = new
    return c_oa - c_nv


total = particle + gas
print(f"model SOA at t = {base.time_h[time_index]:g} h: {c_oa - c_nv:.2f} µg/m³")
for shift in (-1, 0, 1, 2):   # Try this: other shifts, e.g. 0.5 or -2
    print(f"all c* × 10^{shift:+d}:  equilibrium SOA = {equilibrium_soa(total, log10_cstar, c_nv, shift):6.2f} µg/m³")

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; <mark>Shifting every c\* up by one decade (what a method that is one decade too volatile for every product would do; errors that differ from molecule to molecule partly cancel) removes about three quarters of the SOA</mark>: the equilibrium SOA falls from 8.8 to 2.4 µg m⁻³. Two decades remove almost all of it (0.2 µg m⁻³), and one decade down raises it by about 70 % (15.2 µg m⁻³).

### Where does the SOA go?

The number size distribution shows the seed growing as SOA condenses. The composition plot shows
the dry mass by component. Sulfate grows only a little (H₂SO₄ from SO₂ + OH), and there is no
nitrate because N₂O₅ uptake is off in this level.

In [ ]:
plots.size_distribution(base)   # Try this: quantity="mass"; or plots.banana(base)

In [ ]:
plots.composition(base)   # Try this: fraction=True

## 5. Top SOA contributors

The model keeps track of how much of each product is in the particles, so it can list the species
that make up the SOA. The bars are coloured by whether the formula contains N. The title gives the
N-containing share of the mass shown.

In [ ]:
plots.top_contributors(base)   # Try this: n=5

> <font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; Most of the SOA here is hydroperoxides (–OOH), and only a small part contains nitrogen:
>
> 1. The main hydroperoxides carry five O atoms each, spread over several functional groups, and have low c\*: LIMALOOH 10^0.14 µg m⁻³, C826OOH 10^0.61, C728OOH 10^−0.26 and C825OOH 10^−0.74. At a C_OA near 9 µg m⁻³ they are mostly in the particles. Species with OOH in their name make up about 90 % of the SOA, and LIMALOOH alone about 43 %.
> 2. The table makes most nitrogen-containing products too volatile to matter. Only 23 of the 54 PANs condense at all; C822PAN, for example, has 10^4.85 µg m⁻³ and stays in the gas. Nitrogen-containing species make up about 5 % of the SOA (3 % of the bars above).
> 3. A product made in large amounts can still stay in the gas. Limonaldehyde (10^5.8 µg m⁻³) is above the top class and never condenses. Limononic acid (10^3.5) is condensable, but its particle fraction is only $1/(1 + 10^{3.5}/9.4) \approx 0.3$ %.
>
> These values come from one estimation method, and a one-decade error in every c\* changes the SOA almost fourfold (Section 4). The mechanism matters as much: MCM v3.3.1 has no autoxidation of peroxy radicals and no RO₂ + RO₂ reactions that form dimers, which in laboratory experiments make extremely low-volatility products from monoterpenes with a ring double bond, such as limonene ([Jokinen et al., 2015](https://doi.org/10.1073/pnas.1423977112)), and the model has no reactions in the particles. So "mostly hydroperoxides" describes this model, not necessarily real limonene SOA.

### Under the hood: change c\* yourself

`at.run(..., cstar_ugm3={species: c*})` changes the c\* of the species you list (c\* in µg m⁻³ at
298 K). Every other species keeps its value from the table. Behind the scenes it writes a copy of the
table with your values in it, a CSV with columns `species,cstar_ugm3`, and passes it to the model. A
result made like this can be rerun with `at.run(result)` like any other. `at.run(..., cstar_table=None)`
uses the Li et al. (2016) estimate for every species instead.

**Experiment: what if PANs did not condense?** With the composition-only estimate, PANs made most
of the SOA. Give every condensable PAN c\* = 10⁵ µg m⁻³. That is above the top class, so they stay
in the gas phase. The printed *c\* source* counts the species of your table that condense; the PANs
are no longer among them.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; The comments in the cell below list other changes: keep every hydroperoxide in the gas, make all N-containing products two decades more volatile, make limononic acid low-volatility, or use the Li et al. (2016) estimate for every species. Use one of them and run the cell again.

In [ ]:
pans = [s for s in classes.species if "PAN" in s]
print(f"{len(pans)} condensable PAN species, e.g. {', '.join(pans[:6])}")

new_cstar = {s: 1e5 for s in pans}
# Try this instead: keep every hydroperoxide in the gas
# new_cstar = {s: 1e5 for s in classes.species if "OOH" in s}
# or all N-containing products two decades more volatile
# new_cstar = {s: 10 ** (log_cstar[s] + 2) for s in classes.species if has_n[s]}
# or make limononic acid low-volatility: new_cstar = {"LIMONONIC": 0.1}

changed = at.run(PRESET, cstar_ugm3=new_cstar, name="changed c*")
# or the Li et al. (2016) estimate for every species:
# changed = at.run(PRESET, cstar_table=None, name="Li et al. (2016) c*")
print("c* source:", changed.diagnostics["cstar_source"])
print(f"SOA after 6 h: preset {base.totals['SOA_ugm3'][-1]:.2f}, changed c* {changed.totals['SOA_ugm3'][-1]:.2f} µg/m³")
plots.time_series([base, changed], ["SOA_ugm3", "dry_ugm3"])

In [ ]:
plots.top_contributors(changed)

<details>
<summary><font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; (open after running)</summary>

- **PANs in the gas:** the SOA hardly changes (8.66 → 8.57 µg m⁻³, −1 %). The table already keeps
  most PANs in the gas, and the 23 that condense hold about 1 % of the SOA. With the
  composition-only estimate the same experiment cut the SOA from about 16 to under 1 µg m⁻³.
- **Hydroperoxides in the gas:** the SOA falls from 8.66 to 0.16 µg m⁻³, about fifty times less.
  What is left is a mix of hydroxy compounds, nitrates and acids (C825OH, LIMALNO3, C734OH, LIMONIC).
- **Li et al. (2016) for every species:** 15.9 µg m⁻³, and 99 % of the top species contain nitrogen
  (C822PAN, C823PAN, C923PAN, …).

Which molecules make the SOA, and how much there is, depends on their c\*.

</details>

## 6. Experiments: change one input, compare runs

Each run takes about 6–8 s. `at.run(PRESET, gases_ppb={...})` changes only the gases you list;
the others keep their preset values. Use `name=` to label a run in the plots.

### 6a. NOx: 0, 1 and 10 ppb

The preset has 2.5 ppb NOx (0.5 NO + 2 NO₂). Here NOx is split 20% NO and 80% NO₂.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `NOX_PPB`, e.g. add 2.5 (the preset) or 30. `NO_SHARE` is the NO share of NOx at the start.

In [ ]:
#@title Choose NOx and run
# ppb of NOx at the start (preset: 2.5)
NOX_PPB = [0, 1, 10]  #@param {type:"raw"}
# NO / NOx at the start
NO_SHARE = 0.2  #@param {type:"raw"}

nox_runs = [at.run(PRESET, name=f"NOx = {x:g} ppb",
                   gases_ppb={"NO": NO_SHARE * x, "NO2": (1 - NO_SHARE) * x})
            for x in NOX_PPB]

k_ro2 = {}   # LIMAO2 + HO2 and LIMAO2 + NO (both branches) [cm3 molec-1 s-1]
for r in mech.reactions:
    if "LIMAO2" in r.reactants and len(r.reactants) == 2:
        partner = r.reactants[1 - r.reactants.index("LIMAO2")]
        k_ro2[partner] = k_ro2.get(partner, 0.0) + chem.k[r.label]


def ho2_share(result):
    """Share of the peroxy radicals that react with HO2 rather than NO, from 6-hour means."""
    t = np.asarray(result.time_h)
    ho2 = np.trapezoid(at.series(result, "HO2"), t) / t[-1]                  # [cm-3]
    no = np.trapezoid(at.series(result, "NO"), t) / t[-1] * ppb_to_cm3       # [ppb] -> [cm-3]
    return k_ro2["HO2"] * ho2 / (k_ro2["HO2"] * ho2 + k_ro2["NO"] * no)


for r in nox_runs:
    print(f"{r.scenario['name']:>15}: SOA after 6 h = {r.totals['SOA_ugm3'][-1]:6.2f} µg/m³, "
          f"N-containing share of top species = {100 * at.nitrogen_share(r):3.0f}%, "
          f"RO2 + HO2 share = {ho2_share(r):.2f}")
plots.time_series(nox_runs, ["SOA_ugm3", "dry_ugm3"])

In [ ]:
plots.time_series(nox_runs, ["LIMONENE", "OH", "NO2", "O3"], log_y=True)

<details>
<summary><font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; (open after running)</summary>

- **NOx = 0:** limonene is still oxidized (mostly by O₃), and the SOA is 8.5 µg m⁻³, about as much
  as in the preset (8.7). Chamber studies of limonene ozonolysis also find substantial SOA at low
  NOx, and that the route to it depends strongly on NOx ([Zhang et al., 2006](https://doi.org/10.1021/jp062836f)).
- **NOx = 10 ppb:** the SOA falls to about 1.0 µg m⁻³. High NOx suppresses SOA here: the
  hydroperoxides form from RO₂ + HO₂, and NO competes for the same peroxy radicals. The mean OH is
  also three times the preset's (2.3×10⁶ vs 7.8×10⁵ cm⁻³), so the hydroperoxides that do form are
  destroyed faster in the gas.

The last column printed above is the share of the limonene peroxy radicals that react with HO₂
rather than NO:

$$f_{HO_2} = \frac{k_{HO_2}[\mathrm{HO_2}]}{k_{HO_2}[\mathrm{HO_2}] + k_{NO}[\mathrm{NO}]}$$

$k_{HO_2}$ = 2.1×10⁻¹¹ and $k_{NO}$ = 9.0×10⁻¹² cm³ molec⁻¹ s⁻¹ are the mechanism's rate constants
for LIMAO₂ (for NO, the sum of the two branches, one giving an alkoxy radical and one an organic
nitrate), and [HO₂] and [NO] are the 6-hour means of each run.
$f_{HO_2}$ is 1 without NOx, about 0.5 at 1 ppb, 0.33 in the preset and 0.04 at 10 ppb. It leaves
out the reactions of RO₂ with other RO₂ and with NO₃.

- **Between 0 and 2.5 ppb** the SOA barely changes, although $f_{HO_2}$ falls. More NOx also means
  more OH and HO₂ here, so limonene and its first products react faster (limonene left after 3 h:
  0.54 ppb without NOx, 0.20 ppb in the preset), which partly makes up for the smaller share.
- The N-containing share of the top species rises with NOx (0, 1 and 18 % at 0, 1 and 10 ppb) but
  stays a minority. For 0, 0.5, 1, 2.5, 5, 10 and 20 ppb of NOx the SOA is 8.5, 8.1, 8.2, 8.7, 6.5,
  1.0 and 0.16 µg m⁻³.

</details>

### 6b. Ozone

O₃ removes most of the limonene (Section 2). What happens with half or double the O₃?

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `O3_PPB`, e.g. to 10 or 160 ppb, or hold O₃ constant with `hold_constant=["O3"]`.

In [ ]:
#@title Choose the O₃ levels and run
# ppb at the start (preset: 40)
O3_PPB = [20, 80]  #@param {type:"raw"}

o3_runs = [base] + [at.run(PRESET, name=f"O₃ = {x:g} ppb", gases_ppb={"O3": x}) for x in O3_PPB]
labels = ["O₃ = 40 ppb (preset)"] + [f"O₃ = {x:g} ppb" for x in O3_PPB]
for r, label in zip(o3_runs, labels):
    print(f"{label:>22}: limonene left after 3 h = {at.value_at(r, 'LIMONENE', 3.0):.3f} ppb, "
          f"SOA after 6 h = {r.totals['SOA_ugm3'][-1]:5.2f} µg/m³")
plots.time_series(o3_runs, ["LIMONENE", "OH", "SOA_ugm3"], labels=labels, log_y=True)

<details>
<summary><font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; (open after running)</summary>

More O₃ removes limonene faster (0.59, 0.20 and 0.015 ppb left after 3 h at 20, 40 and 80 ppb O₃),
and it also raises OH (ozonolysis releases OH, and O₃ photolysis makes more). The SOA rises with
O₃ too: from 7.2 to 11.6 µg m⁻³ between 20 and 80 ppb, and from 6.4 to 17.1 µg m⁻³ between 10 and
160 ppb. By 6 hours nearly all of the limonene has reacted in every one of these runs, so the extra
SOA comes from the later steps. More O₃ turns more NO into NO₂, so a larger share of the peroxy
radicals reacts with HO₂ and makes hydroperoxides: $f_{HO_2}$ (Section 6a) is about 0.27 at 20 ppb
O₃, 0.33 in the preset and 0.42 at 80 ppb. More OH also turns more limonaldehyde into LIMALO₂ and
then into LIMALOOH (Section 2), which rises from 3.3 to 4.6 µg m⁻³ between 20 and 80 ppb.

</details>

### 6c. Initial limonene

The mass yield is $Y = \Delta\mathrm{SOA} / \Delta\mathrm{LIM}$, both in µg m⁻³. To convert
ppb to µg m⁻³: $C = \chi\,[M]\,\mathrm{MW}/N_A$ (with unit factors).

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `LIMONENE_PPB`, e.g. to 2 or 40 ppb.

In [ ]:
#@title Choose the limonene levels and run
# ppb at the start (preset: 10)
LIMONENE_PPB = [5, 20]  #@param {type:"raw"}

lim_runs = [at.run(PRESET, name=f"limonene = {x:g} ppb", gases_ppb={"LIMONENE": x}) for x in LIMONENE_PPB]
lim_runs = sorted([base] + lim_runs, key=lambda r: r.scenario["gases_ppb"]["LIMONENE"])

ugm3_per_ppb = at.ugm3_per_ppb(mech.molecular_weight("LIMONENE"), base)   # molar mass in g/mol
print(f"1 ppb limonene = {ugm3_per_ppb:.2f} µg/m³ at 298 K, 1 atm\n")
print("limonene  reacted   SOA    yield")
for r in lim_runs:
    lim = at.series(r, "LIMONENE")
    reacted = (lim[0] - lim[-1]) * ugm3_per_ppb
    soa = r.totals["SOA_ugm3"][-1]
    print(f"{lim[0]:5.0f} ppb  {reacted:5.1f}  {soa:6.2f}  {soa / reacted:6.2f}   (µg/m³)")
plots.time_series(lim_runs, ["SOA_ugm3", "OH"])

<details>
<summary><font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; (open after running)</summary>

The SOA grows faster than the limonene: from 5 to 20 ppb limonene the SOA goes from 3.0 to
17.8 µg m⁻³ (six times more), and the yield rises from 0.11 to 0.16. Much of this is the classic rise of
yield with organic loading ([Odum et al., 1996](https://doi.org/10.1021/es950943+)). If the reacted limonene makes products $i$ with
mass yields $\alpha_i$ (µg of product per µg of limonene) and saturation concentrations $c^*_i$,
absorptive partitioning gives

$$Y = \sum_i \frac{\alpha_i}{1 + c^*_i / C_{OA}}$$

More limonene makes more C_OA, so a larger fraction of each semi-volatile product condenses. The
rise flattens once C_OA is well above the c\* of the main products (about 0.2–4 µg m⁻³ for the four
top hydroperoxides).

The $\alpha_i$ are not fixed, though. Less limonene uses up less OH, and NO stays higher: at 1 ppb the
mean OH is about 5.6×10⁶ cm⁻³ instead of 7.8×10⁵, and $f_{HO_2}$ (Section 6a) is about 0.08 instead of
0.33. So far fewer hydroperoxides form, and OH destroys more of them in the gas. For 1, 2, 5, 10, 20
and 40 ppb of limonene the yield is 0.008, 0.021, 0.11, 0.16, 0.16 and 0.17. With the products made in
the same proportions as at 10 ppb, partitioning alone would give about 0.09 at 1 ppb and 0.13 at 5 ppb.
So below about 5 ppb the low yield comes mostly from the chemistry, not from partitioning.

</details>

### 6d. Seed loading

Change the number of seed particles (same size and composition). The seed carries 30% non-volatile
organic, so a bigger seed also brings more absorbing mass at the start.

> <font color="#A47F52"><b>TRY THIS</b></font> &nbsp; Change `SEED_NUMBER_CM3`, e.g. to 100 or 30000; or add `"gmd_nm"` to the seed dict.

In [ ]:
#@title Choose the seed numbers and run
# particles per cm³ (preset: 2000)
SEED_NUMBER_CM3 = [500, 10000]  #@param {type:"raw"}

seed_runs = [at.run(PRESET, name=f"seed {n:g} cm⁻³", seed={"number_cm3": n}) for n in SEED_NUMBER_CM3]
seed_runs = sorted([base] + seed_runs, key=lambda r: r.scenario["seed"]["modes"][0]["number_cm3"])
for r in seed_runs:
    s = at.summary(r)
    print(f"seed {r.scenario['seed']['modes'][0]['number_cm3']:6.0f} cm⁻³: POA = {s['POA_ugm3_start']:.2f}, "
          f"SOA after 6 h = {s['SOA_ugm3_end']:5.2f} µg/m³, GMD {s['gmd_nm_start']:.0f} → {s['gmd_nm_end']:.0f} nm")
plots.compare_size_distributions(seed_runs)

<details>
<summary><font color="#858F6A"><b>WHAT TO NOTICE</b></font> &nbsp; (open after running)</summary>

The SOA changes by only about 13 % (8.4 to 9.5 µg m⁻³) while the number of particles changes
20-fold. What changes is how the SOA is shared out. A few particles grow a lot (to about 270 nm
with 500 cm⁻³), and many particles grow a little (to about 115 nm with 10 000 cm⁻³). Each run
starts from its own seed (dotted lines). The seed's organic (0.2–3.8 µg m⁻³) adds absorbing mass,
which is why the largest seed makes the most SOA. But about 85 % of the particle SOA sits in classes
with c\* of 3 µg m⁻³ or less, which are almost 90 % condensed at this C_OA, so extra absorbing mass
mostly moves the more volatile classes. Do not generalize this: when the organic loading is low
or the products are more volatile, the absorbing mass matters a lot. In Level 1, try c\* = 10 with
a 2000 cm⁻³, 150 nm seed: the SOA rises about tenfold (0.2 → 2.5 µg m⁻³).

</details>

## 7. Run it yourself

> <font color="#A47F52"><b>YOUR TURN</b></font> &nbsp; Pick limonene, NOx and O₃ and press **Run model** (about 6–10 s). The plots compare your last three runs with the preset. Every run is kept in the list `widget_runs`, so you can use it with any plot function afterwards, e.g. `plots.top_contributors(widget_runs[-1])`.

In [ ]:
import html

import ipywidgets as widgets
from IPython.display import clear_output, display

WIDGET_NO_SHARE = 0.2   # NO / NOx at the start, as in Section 6a
limonene_in = widgets.SelectionSlider(options=[1, 2, 5, 10, 20, 40], value=10, description="Limonene (ppb)")
nox_in = widgets.SelectionSlider(options=[0, 0.5, 1, 2.5, 5, 10, 20], value=2.5, description="NOx (ppb)")
o3_in = widgets.SelectionSlider(options=[10, 20, 40, 80, 160], value=40, description="O₃ (ppb)")
for slider in (limonene_in, nox_in, o3_in):
    slider.style.description_width = "110px"
    slider.layout.width = "420px"
run_button = widgets.Button(description="Run model", button_style="primary", icon="play")
status = widgets.HTML("Pick values and press <b>Run model</b>.")
output = widgets.Output()
widget_runs = []


def on_run(_):
    run_button.disabled = True
    status.value = "Running… (about 6–10 s)"
    lim, nox, o3 = limonene_in.value, nox_in.value, o3_in.value
    try:
        result = at.run(PRESET, progress=False, name=f"lim {lim:g}, NOx {nox:g}, O₃ {o3:g} ppb",
                        gases_ppb={"LIMONENE": lim, "O3": o3,
                                   "NO": WIDGET_NO_SHARE * nox, "NO2": (1 - WIDGET_NO_SHARE) * nox})
    except Exception as err:   # show the error here; widget callbacks otherwise hide it
        status.value = f"<b>Run failed:</b> {html.escape(str(err))}"
        return
    finally:
        run_button.disabled = False
    widget_runs.append(result)
    status.value = (f"Done in {result.diagnostics['runtime_s']:.0f} s. SOA after 6 h: "
                    f"<b>{result.totals['SOA_ugm3'][-1]:.2f} µg m⁻³</b>")
    with output:
        clear_output(wait=True)
        recent = [base] + widget_runs[-3:]
        plots.show_in_widget(plots.time_series(recent, ["SOA_ugm3", "dry_ugm3"]),
                             plots.time_series(recent, ["LIMONENE", "OH", "NO2"], log_y=True),
                             plots.top_contributors(result))


run_button.on_click(on_run)
display(widgets.VBox([limonene_in, nox_in, o3_in, widgets.HBox([run_button, status]), output]))

## Questions to think about

> 1. More NOx lowers the SOA, and more O₃ raises it. Which reactions of the peroxy radicals explain each?
> 2. Which properties of a product decide its particle fraction? Which of them does a group-contribution method such as Nannoolal et al. (2008) use that the Li et al. (2016) formula does not?
> 3. Most PANs stay in the gas here, but 23 still condense. PANs decompose in the gas phase within about 40 minutes. What could happen to a PAN in a real particle, and what does the model assume?
> 4. Repeat the NOx experiment (6a) and the limonene experiment (6c) with the Li et al. (2016) estimate for every species (add `cstar_table=None` to the runs). Which conclusions change?
> 5. The seed experiment changed particle number 20-fold but SOA by only about 13 %. When would the seed's organic mass matter more?
> 6. Light is constant here. Which results would change under real sunlight with a night (Level 3)? Remember that OH destroys the main SOA species in the gas phase (Section 2).

## Going further

**The raw model API.** `at.run` is a thin wrapper. Underneath, a run is a `Scenario` (frozen
dataclasses, loaded from a TOML preset) passed to `tomas_jax.box.driver.run`. Two differences: the
raw `Scenario.replace(gases_ppb=...)` replaces the *whole* gas dictionary, while `at.run` merges it
into the preset's gases. And `at.run(PRESET)` uses `N_BINS` size bins (set at the top, 40 by default), while `load_preset` always keeps the preset's
40 (as the web app does).

In [ ]:
from tomas_jax.box import driver, scenario

s = scenario.load_preset(PRESET).replace(duration_h=1.0, photolysis={"jno2_per_s": 8e-3})
print(s.photolysis)
raw = driver.run(s)   # what at.run calls, without the progress bar and timeout
print(f"{raw.diagnostics['runtime_s']:.1f} s; SOA after 1 h = {raw.totals['SOA_ugm3'][-1]:.2f} µg/m³")

**More things to try**

- Dimmer or brighter light: `at.run(PRESET, photolysis={"jno2_per_s": 2e-3})` (clear-sky midday
  summer sun gives about 1×10⁻² s⁻¹ in TUV-x; see Level 3).
- Hold a gas at its initial value: `at.run(PRESET, hold_constant=["O3"])` or `["NO2"]` (a
  stand-in for continuous emissions).
- Temperature: `environment={"temp_k": 288}`. Every c\* changes with T by Clausius–Clapeyron
  (ΔH_vap = 131 − 11 log₁₀ c\* kJ mol⁻¹).
- Seed composition: `seed={"organic_mass_fraction": 0.9}`.
- Save a run and load it later: `at.save_result(r, "my_run.json")`, `at.load_result("my_run.json")`.
- See exactly what a run will use without running it: `at.describe(at.scenario_from(PRESET, gases_ppb={"NO": 0}))`.
- Read more: [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/0e23da0da79a5a080baa02447e47951b75f796fe/ASSUMPTIONS.md) (A1–A14),
  [gaschem.md](https://github.com/reflective-org/TOMAS-JAX/blob/0e23da0da79a5a080baa02447e47951b75f796fe/docs/aaar/gaschem.md) (mechanism, solver, photolysis, volatility) and
  [soa_partitioning.md](https://github.com/reflective-org/TOMAS-JAX/blob/0e23da0da79a5a080baa02447e47951b75f796fe/docs/aaar/soa_partitioning.md) (the kinetic partitioning).
- Back to partitioning with a single surrogate: [Level 1 · Volatility](01_volatility.ipynb).
- Next: [Level 3 · Day & Night](03_day_and_night.ipynb) adds the real diurnal cycle of sunlight,
  night-time NO₃ chemistry and N₂O₅ uptake. [Inside the model](04_inside_the_model.ipynb) goes
  deeper into the code.

### References

<details>
<summary>11 references (click to show)</summary>

- Jenkin, M. E., Saunders, S. M., and Pilling, M. J. (1997), Atmos. Environ. 31, 81–104, [doi:10.1016/S1352-2310(96)00105-7](https://doi.org/10.1016/S1352-2310%2896%2900105-7); Saunders,
  S. M., et al. (2003), ACP 3, 161–180, [doi:10.5194/acp-3-161-2003](https://doi.org/10.5194/acp-3-161-2003), [PDF](https://acp.copernicus.org/articles/3/161/2003/acp-3-161-2003.pdf): the Master Chemical Mechanism (https://mcm.york.ac.uk).
- Jokinen, T., et al. (2015), Production of extremely low volatile organic compounds from biogenic
  emissions: measured yields and atmospheric implications, PNAS 112, 7123–7128.
  [doi:10.1073/pnas.1423977112](https://doi.org/10.1073/pnas.1423977112): the autoxidation products MCM v3.3.1 leaves out.
- Li, Y., Pöschl, U., and Shiraiwa, M. (2016), Molecular corridors and parameterizations of
  volatility in the chemical evolution of organic aerosols, ACP 16, 3327–3344. [doi:10.5194/acp-16-3327-2016](https://doi.org/10.5194/acp-16-3327-2016), [PDF](https://acp.copernicus.org/articles/16/3327/2016/acp-16-3327-2016.pdf): the comparison c\* estimate.
- Nannoolal, Y., Rarey, J., and Ramjugernath, D. (2008), Estimation of pure component properties. Part 3.
  Estimation of the vapor pressure of non-electrolyte organic compounds via group contributions and group
  interactions, Fluid Phase Equilib. 269, 117–133. [doi:10.1016/j.fluid.2008.04.020](https://doi.org/10.1016/j.fluid.2008.04.020): the c\* of the products.
- O'Meara, S. P., et al. (2021), PyCHAM (v2.1.1): a Python box model for simulating aerosol chambers,
  GMD 14, 675–702. [doi:10.5194/gmd-14-675-2021](https://doi.org/10.5194/gmd-14-675-2021): the model that computed the table's vapour pressures.
- Pankow, J. F. (1994), Atmos. Environ. 28, 185–188, [doi:10.1016/1352-2310(94)90093-0](https://doi.org/10.1016/1352-2310%2894%2990093-0); Donahue, N. M., et al. (2006), ES&T 40,
  2635–2643, [doi:10.1021/es052297c](https://doi.org/10.1021/es052297c): absorptive partitioning and the volatility basis set.
- Odum, J. R., et al. (1996), ES&T 30, 2580–2585: SOA yields and organic mass. [doi:10.1021/es950943+](https://doi.org/10.1021/es950943+)
- Epstein, S. A., Riipinen, I., and Donahue, N. M. (2010), ES&T 44, 743–748: ΔH_vap(c\*). [doi:10.1021/es902497z](https://doi.org/10.1021/es902497z)
- Zhang, J., Huff Hartz, K. E., Pandis, S. N., and Donahue, N. M. (2006), Secondary organic aerosol
  formation from limonene ozonolysis: homogeneous and heterogeneous influences as a function of NOx,
  J. Phys. Chem. A 110, 11053–11063. [doi:10.1021/jp062836f](https://doi.org/10.1021/jp062836f): limonene SOA at low and high NOx.

</details>

---

<a href="https://www.reflective.org"><img src="https://raw.githubusercontent.com/reflective-org/AAAR-2026-Aerosol-Tutorial/9feba6889d498d8c754a6783e5460f46e054d60f/assets/supported-by-reflective.png" width="294" alt="Supported by Reflective"></a>